# Prepare daily input / 準備每日輸入

## Goal / 目標
The three methods use the same frozen daily inflow and physical case. The sample is already aggregated and upstream-routed; insert it once at the reservoir inflow node. It is simulated inflow, not a forecast. Excel users should export a CSV with an explicit date column and units.

![FE Walter](../docs/figures/fe-walter-study-map.png)

三方法共用固定 daily 入流與物理 case。此資料已彙整並完成上游 routing，只在水庫入流節點加入一次；它是模擬入流，不是預報。Excel 請匯出含日期與明確單位的 CSV。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Read three daily inputs / 讀取三日輸入

In [2]:
import csv
rows=list(csv.DictReader((ROOT/'cases/fe_walter_reference/inflows.csv').open()))
print(rows[:3])
case=json.loads((ROOT/'cases/fe_walter_reference/case.json').read_text())
print('Physical settings:', case['network']['reservoirs']['FEWalter']['physical'])

[{'date': '2016-01-01', 'node': 'FEW_INFLOW', 'local_inflow_cms': '32.12531870666853'}, {'date': '2016-01-01', 'node': 'Downstream', 'local_inflow_cms': '0.0'}, {'date': '2016-01-02', 'node': 'FEW_INFLOW', 'local_inflow_cms': '22.353047216043638'}]
Physical settings: {'reservoir_id': 'fe_walter', 'capacity_m3': 135532852.33895192, 'dead_storage_m3': 49.3392742128, 'initial_storage_m3': 2457848.279729185, 'minimum_release_cms': 1.41584235, 'maximum_release_cms': 283.16847, 'dt_seconds': 86400.0}


### Check dates and units / 檢查日期與單位

In [3]:
settings=load_settings(ROOT/'examples/fe_walter/zero_shot/experiment.hydroopslm.offline.yaml')
print({k:settings.report[k] for k in ['days','start_date','end_date','data_kind','live_ready']})
print('Map units:', [f['properties']['subbasin_id'] for f in json.loads((ROOT/'data/figures/fe_walter_map.json').read_text())['subbasins']])

{'days': 3, 'start_date': '2016-01-01', 'end_date': '2016-01-03', 'data_kind': 'simulated_local_inflow', 'live_ready': False}
Map units: ['FEW-D-01', 'FEW-D-02', 'FEW-X-01', 'FEW-X-02', 'FEW-X-03', 'FEW-R-01', 'FEW-R-02']


## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
See docs/HYDROLOGY_SETUP.md for native basin areas, rainfall-runoff parameters, routing and C&V. Do not add a second copy of already routed inflow.

原生面積、水文參數、routing 與 C&V 見 docs/HYDROLOGY_SETUP.md。不要重複加入已 routing 入流。